# Connection Pooling, Backpressure, Idempotency & Dead Letter Queue

> **Four patterns that prevent a working system from becoming an unworking one
> under load, partial failures, or message delivery uncertainties.**


---
## Connection Pooling

### 🧠 Mental Model

> **A connection pool is a taxi rank. Instead of every passenger hailing a new taxi
> (expensive: time + cost), they pick from a queue of waiting taxis (instantly available).
> Taxis return to the rank when done. Total taxis = pool size = how many simultaneous
> passengers can be served.**

### WHY It Exists — The Cost of Creating Connections

```
TCP connection: ~1ms (3-way handshake)
TLS handshake:  ~10ms (adds on top of TCP)
Database auth:  ~5ms (credential verification, session setup)
Total per-connection: ~16ms overhead per request!

At 1,000 RPS:
  WITHOUT pool: 1,000 × 16ms overhead = 16 seconds of wasted CPU per second
  WITH pool:    connections reused; near-zero overhead per request

Database connection also consumes server resources:
  PostgreSQL: ~5MB per connection × 100 connections = 500MB RAM just for connections!
  Pool limits this to a fixed maximum.
```

### ❌ Without Connection Pooling

```
ShopFlow's Black Friday: 5,000 concurrent users each trigger a DB query.
Without pool: 5,000 new DB connections.
PostgreSQL max_connections default: 100.
Connections beyond 100: REFUSED → HTTP 500 errors → cascading failures.
DB server OOM: RAM for 5,000 connections would be 25GB.
```

### ✅ With Connection Pooling (PgBouncer, SQLAlchemy pool, HikariCP)

```
Pool size: 20 DB connections (fixed)
Concurrent users: 5,000
Queue: requests wait (milliseconds) for a connection to be released
Result: DB gets at most 20 concurrent queries → stable under any load
```

### Pool Sizing Formula

```
Connections = Threads × (1 + waiting_time / service_time)

For PostgreSQL:
  optimal_connections ≈ num_CPU_cores × 2 + num_disks

Rules of thumb:
  OLTP (fast queries): pool_size = 10-20 per app server
  Analytics (slow queries): pool_size = 5-10 per app server
  Total connections < DB max_connections / 2 (leave headroom)
```


In [ ]:
import threading, time, queue, random
from contextlib import contextmanager
from dataclasses import dataclass

@dataclass
class Connection:
    conn_id: int
    in_use:  bool = False
    queries: int  = 0

    def execute(self, sql: str) -> dict:
        self.queries += 1
        time.sleep(random.uniform(0.001, 0.005))   # simulate DB latency
        return {"result": f"conn-{self.conn_id} executed: {sql[:30]}"}

class ConnectionPool:
    '''Thread-safe database connection pool.'''

    def __init__(self, max_size: int, timeout: float = 5.0):
        self.max_size  = max_size
        self.timeout   = timeout
        self._pool: queue.Queue[Connection] = queue.Queue(maxsize=max_size)
        self._created  = 0
        self._lock     = threading.Lock()
        self.wait_times: list[float] = []
        self.timeouts: int = 0

        # Pre-create minimum connections
        for _ in range(max(2, max_size // 2)):
            self._create_connection()

    def _create_connection(self) -> Connection | None:
        with self._lock:
            if self._created >= self.max_size: return None
            self._created += 1
            conn = Connection(self._created)
            self._pool.put(conn)
            return conn

    @contextmanager
    def acquire(self):
        '''Borrow a connection from the pool; return it automatically.'''
        t0   = time.perf_counter()
        conn = None
        try:
            try:
                conn = self._pool.get(timeout=self.timeout)
            except queue.Empty:
                # Pool exhausted — try creating a new one
                conn = self._create_connection()
                if conn is None:
                    self.timeouts += 1
                    raise TimeoutError(f"Pool exhausted (size={self.max_size}); no connection available")
                # Remove from queue since we just created it directly
                self._pool.get_nowait() if not self._pool.empty() else None
                conn = Connection(self._created)

            wait_ms = (time.perf_counter() - t0) * 1000
            self.wait_times.append(wait_ms)
            conn.in_use = True
            yield conn
        finally:
            if conn:
                conn.in_use = False
                try:
                    self._pool.put_nowait(conn)
                except queue.Full:
                    pass  # pool is full, discard the extra connection

# ── Demo: pool of 5 handling 20 concurrent requests ─────────────────────────
pool = ConnectionPool(max_size=5)
results = []
errors  = []

def run_query(request_id: int):
    try:
        with pool.acquire() as conn:
            result = conn.execute(f"SELECT * FROM orders WHERE id={request_id}")
            results.append(result)
    except TimeoutError as e:
        errors.append(str(e))

threads = [threading.Thread(target=run_query, args=(i,)) for i in range(20)]
t0 = time.perf_counter()
for t in threads: t.start()
for t in threads: t.join()
elapsed = time.perf_counter() - t0

print(f"=== Connection Pool Demo ===")
print(f"Pool size:        {pool.max_size}")
print(f"Concurrent reqs:  20")
print(f"Successful:       {len(results)}")
print(f"Timeouts:         {len(errors)}")
print(f"Total time:       {elapsed*1000:.0f}ms")
if pool.wait_times:
    avg_wait = sum(pool.wait_times) / len(pool.wait_times)
    print(f"Avg wait for conn:{avg_wait:.2f}ms")
print(f"Total DB queries: {sum(c.queries for c in [pool._pool.get_nowait() for _ in range(pool._pool.qsize())] if True):}")
print(f"
KEY INSIGHT: 5 connections handled 20 concurrent requests efficiently.")
print(f"Without a pool: 20 new connections would be created (each costs ~16ms).")


---
## Backpressure

### 🧠 Mental Model

> **Backpressure = a garden hose with your thumb over the end. You're slowing down
> the source (water) because the output can't handle full flow. Without it, the hose
> overflows. With it, the flow is controlled to match what downstream can handle.**

### WHY Backpressure Exists

```
Without backpressure:
  Producer generates 10,000 messages/sec
  Consumer processes  1,000 messages/sec
  Queue:  1,000 messages/sec overflow
  After 1 hour: 3.6M backlogged messages → queue OOM → system crashes

With backpressure:
  Consumer signals "I'm at 80% capacity"
  Producer slows to 1,000 messages/sec (matches consumer)
  Queue stays bounded → stable indefinitely
```

### Backpressure Strategies

```
1. BLOCK/SLOW (simplest):
   Producer blocks when queue is full.
   asyncio.Queue(maxsize=100) → await queue.put() blocks when full
   ✓ Simple    ✗ Propagates delay upstream

2. DROP (fastest, lossy):
   Reject new items when at capacity.
   Good for: metrics, non-critical events, real-time streams
   ✗ Data loss   ✓ Never blocks

3. SAMPLE (rate limiting):
   Accept only 1 in N items when overloaded.
   Good for: telemetry, logging, monitoring

4. BUFFER + DRAIN (async):
   Accept all items into a bounded buffer.
   Drain buffer at consumer's pace.
   asyncio.Semaphore(max_concurrent) controls parallelism
```

### 🌍 Where Backpressure is Used in Production

| System | Backpressure mechanism |
|---|---|
| Kafka consumer | `max.poll.records` + consumer lag monitoring |
| gRPC streaming | Flow control built into HTTP/2 (window-based) |
| Reactive Streams | `Publisher.request(n)` — consumer pulls N at a time |
| TCP | Receive window size limits sender speed |
| NGINX | `limit_req_zone` — queue + reject when full |


In [ ]:
import asyncio, time, random
from collections import deque

class BackpressureQueue:
    '''Bounded queue with configurable backpressure strategy.'''

    def __init__(self, max_size: int, strategy: str = "block"):
        self.max_size = max_size
        self.strategy = strategy  # "block", "drop", "sample"
        self._q: asyncio.Queue = asyncio.Queue(maxsize=max_size)
        self.dropped = 0
        self.sampled = 0
        self.processed = 0

    async def put(self, item) -> bool:
        '''Returns True if item was accepted.'''
        if self.strategy == "block":
            await self._q.put(item)   # blocks if full
            return True
        elif self.strategy == "drop":
            if self._q.full():
                self.dropped += 1
                return False   # reject
            await self._q.put(item)
            return True
        elif self.strategy == "sample":
            load = self._q.qsize() / self.max_size
            # Accept all when < 50% full; sample down as load increases
            accept_prob = max(0.1, 1.0 - load)
            if random.random() > accept_prob:
                self.sampled += 1
                return False
            await self._q.put(item)
            return True
        return False

    async def get(self):
        item = await self._q.get()
        self.processed += 1
        return item

    @property
    def utilization(self) -> float:
        return self._q.qsize() / self.max_size

async def producer(q: BackpressureQueue, rate_per_sec: int, duration: float):
    '''Generate messages at a specified rate.'''
    interval = 1.0 / rate_per_sec
    end      = asyncio.get_event_loop().time() + duration
    sent = 0
    while asyncio.get_event_loop().time() < end:
        await q.put({"id": sent, "ts": time.time()})
        sent += 1
        await asyncio.sleep(interval)
    return sent

async def consumer(q: BackpressureQueue, rate_per_sec: int, duration: float):
    '''Consume messages at a specified rate (simulates slow processing).'''
    delay = 1.0 / rate_per_sec
    end   = asyncio.get_event_loop().time() + duration
    while asyncio.get_event_loop().time() < end:
        try:
            item = await asyncio.wait_for(q.get(), timeout=0.1)
            await asyncio.sleep(delay)   # simulate processing
        except asyncio.TimeoutError:
            pass

async def run_comparison():
    print("=== Backpressure Strategy Comparison ===")
    print(f"Producer: 5,000 msg/sec   Consumer: 1,000 msg/sec   Queue: 100 slots
")

    for strategy in ["drop", "sample"]:
        q = BackpressureQueue(max_size=100, strategy=strategy)
        # Run 2 seconds of simulation
        duration = 2.0
        p_task = asyncio.create_task(producer(q, rate_per_sec=500, duration=duration))
        c_task = asyncio.create_task(consumer(q, rate_per_sec=100, duration=duration))
        sent = await p_task
        await c_task

        print(f"Strategy: {strategy.upper():8s}")
        print(f"  Sent:      {sent:5d}  |  Processed: {q.processed:5d}")
        print(f"  Dropped:   {q.dropped:5d}  |  Sampled:   {q.sampled:5d}")
        print(f"  Final queue size: {q._q.qsize()} / {q.max_size}")
        print(f"  KEY: queue bounded → system stable despite 5× producer speed
")

asyncio.run(run_comparison())


---
## Idempotency

### 🧠 Mental Model

> **Idempotency = a light switch. Toggle it ON. Toggle it ON again. Still ON.
> The second "toggle" has no additional effect. An idempotent operation can be
> safely retried without producing different results.**

### WHY It Exists — The Double-Payment Problem

```
ShopFlow checkout flow:
  1. Client sends POST /charge (amount=$99)
  2. Server charges Stripe → SUCCESS
  3. Server tries to write order to DB → TIMEOUT (network issue)
  4. Server returns HTTP 500 to client
  5. Client retries POST /charge
  → Stripe charged TWICE! Customer billed $198 for one order.

Without idempotency: retries cause double-charges, duplicate orders, duplicate emails.
With idempotency:   retries are safe — the system recognizes and ignores duplicates.
```

### HOW to Implement Idempotency

```
KEY INSIGHT: Client generates a unique idempotency key for the request.
Server stores the result of the first successful execution.
On retry (same key): return the stored result WITHOUT re-executing.

Steps:
1. Client generates key: idempotency_key = uuid4()
2. Client sends: POST /charge {amount: 99} with header Idempotency-Key: <key>
3. Server checks: have I processed <key> before?
   → Yes: return stored response
   → No:  execute, store (key, response), return response
4. Client retries with same key → gets same stored response, no double-charge
```

### 🌍 Where Idempotency is Used

| System | Implementation |
|---|---|
| Stripe API | Idempotency-Key header; stored 24h |
| AWS API | `ClientToken` parameter on EC2, S3 |
| Shopify | Idempotency key on order creation |
| Twilio | `SmsSid` prevents duplicate sends |
| Database | `INSERT ... ON CONFLICT DO NOTHING` |


---
## Dead Letter Queue (DLQ)

### 🧠 Mental Model

> **A DLQ is the "return to sender" pile on a postal worker's desk. Messages that
> can't be delivered after N attempts go to the DLQ instead of being silently dropped.
> An engineer can later examine them, fix the bug, and re-send them.**

### WHY DLQ Exists

```
Order event sent to fulfilment service:
  Attempt 1: Service unavailable → retry after 1s
  Attempt 2: Service unavailable → retry after 2s
  Attempt 3: Service unavailable → retry after 4s
  Attempt 4: Poison pill (malformed JSON) → processing always fails

WITHOUT DLQ: after max retries, message is DROPPED. Order never fulfilled.
             Engineer never knows. Customer calls support. Revenue lost.

WITH DLQ:    after max retries, message goes to DLQ.
             Alert fires: "DLQ has 1 message"
             Engineer inspects: malformed order ID
             Engineer fixes producer + reprocesses DLQ
             Order fulfilled. Customer happy.
```

### When a Message Goes to DLQ

1. **Poison pill** — malformed message that always fails (bad JSON, invalid ID)
2. **Downstream unavailable** — service down for longer than retry window
3. **Business rule violation** — order for deleted product, payment for banned user
4. **Timeout** — processing takes too long (DB slow, external API hung)

### 🌍 DLQ in Real Systems

| System | DLQ implementation |
|---|---|
| AWS SQS | `RedrivePolicy` → moves to DLQ after maxReceiveCount |
| Kafka | DLQ = separate topic, written to by consumer on failure |
| RabbitMQ | Dead letter exchange + routing key |
| Azure Service Bus | Dead-letter sub-queue per queue |
| Google Pub/Sub | Dead letter topic with subscription |


In [ ]:
import json, time, uuid, threading
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class Message:
    message_id:   str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    body:         dict = field(default_factory=dict)
    attempt:      int = 0
    max_attempts: int = 3

class DeadLetterQueue:
    def __init__(self):
        self._messages: list[Message] = []
        self._lock = threading.Lock()

    def enqueue(self, msg: Message):
        with self._lock:
            self._messages.append(msg)
            print(f"  [DLQ] Message {msg.message_id} sent to DLQ "
                  f"after {msg.attempt} attempts. Body: {msg.body}")

    def reprocess(self, handler: Callable[[Message], bool]) -> int:
        '''Re-run all DLQ messages through the (fixed) handler.'''
        with self._lock:
            msgs = list(self._messages)
            self._messages.clear()
        success = 0
        for msg in msgs:
            msg.attempt = 0   # reset for reprocessing
            if handler(msg): success += 1
        print(f"  [DLQ] Reprocessed {success}/{len(msgs)} messages successfully")
        return success

    @property
    def size(self) -> int:
        return len(self._messages)

class MessageQueue:
    def __init__(self, dlq: DeadLetterQueue, base_delay: float = 0.01):
        self._pending: list[Message] = []
        self._dlq = dlq
        self._base_delay = base_delay

    def enqueue(self, body: dict, max_attempts: int = 3) -> str:
        msg = Message(body=body, max_attempts=max_attempts)
        self._pending.append(msg)
        return msg.message_id

    def process_all(self, handler: Callable[[Message], bool]):
        while self._pending:
            msg = self._pending.pop(0)
            msg.attempt += 1
            success = False
            try:
                success = handler(msg)
            except Exception as e:
                print(f"  [Queue] Message {msg.message_id} ERRORED: {e}")

            if success:
                print(f"  [Queue] Message {msg.message_id} processed ✓ (attempt {msg.attempt})")
            elif msg.attempt >= msg.max_attempts:
                self._dlq.enqueue(msg)
            else:
                delay = self._base_delay * (2 ** (msg.attempt - 1))
                time.sleep(delay)
                self._pending.append(msg)   # retry

# ── Demo ─────────────────────────────────────────────────────────────────────
dlq   = DeadLetterQueue()
queue = MessageQueue(dlq)

# Simulate different message types
print("=== DLQ Demo ===
")
queue.enqueue({"order_id": "ORD-001", "action": "fulfill"})    # will succeed
queue.enqueue({"order_id": None,       "action": "fulfill"})    # poison pill (always fails)
queue.enqueue({"order_id": "ORD-003", "action": "fulfill"})    # will succeed

def fulfil_handler(msg: Message) -> bool:
    order_id = msg.body.get("order_id")
    if order_id is None:
        raise ValueError(f"Invalid order_id: {order_id}")   # poison pill
    if msg.attempt < 2 and order_id == "ORD-001":
        raise ConnectionError("Service temporarily unavailable")  # transient error
    print(f"  [Handler] Fulfilled order {order_id} ✓")
    return True

queue.process_all(fulfil_handler)

print(f"
DLQ size: {dlq.size} message(s)")

# Fix the bug and reprocess DLQ
print("
--- Bug fixed: reprocessing DLQ ---")
def fixed_handler(msg: Message) -> bool:
    order_id = msg.body.get("order_id", "UNKNOWN")
    if order_id == "UNKNOWN" or order_id is None:
        print(f"  [Fixed Handler] Skipping invalid message: {msg.body}")
        return True   # acknowledge to remove from DLQ
    print(f"  [Fixed Handler] Fulfilled: {order_id} ✓")
    return True

dlq.reprocess(fixed_handler)
